# Data Parsing

Load one IPL YAML match file from the extracted 4-season dataset and inspect the main sections.

In [5]:
import yaml

file_path = "../../IPL_Datasets_4_Seasons/1304047.yaml"

with open(file_path, "r") as f:
    data = yaml.safe_load(f)

print(data["info"])
print(data["innings"])


{'balls_per_over': 6, 'city': 'Mumbai', 'competition': 'IPL', 'dates': ['2022-03-26'], 'gender': 'male', 'match_type': 'T20', 'outcome': {'winner': 'Kolkata Knight Riders', 'by': {'wickets': 6}}, 'overs': 20, 'player_of_match': ['UT Yadav'], 'players': {'Chennai Super Kings': ['RD Gaikwad', 'DP Conway', 'RV Uthappa', 'AT Rayudu', 'RA Jadeja', 'S Dube', 'MS Dhoni', 'DJ Bravo', 'MJ Santner', 'AF Milne', 'TU Deshpande'], 'Kolkata Knight Riders': ['AM Rahane', 'VR Iyer', 'N Rana', 'SS Iyer', 'SW Billings', 'SP Jackson', 'AD Russell', 'SP Narine', 'UT Yadav', 'Shivam Mavi', 'CV Varun']}, 'registry': {'people': {'AD Russell': 'bbd41817', 'AF Milne': '350bb1b1', 'AK Chaudhary': 'fdcc6236', 'AM Rahane': '29e95537', 'AT Rayudu': '70d205c9', 'CV Varun': '5b7ab5a9', 'Chirra Ravikanthreddy': '8b680f55', 'DJ Bravo': '87e562a9', 'DP Conway': 'df5a6881', 'M Nayyar': 'b7bccddb', 'MJ Santner': 'e4a0deae', 'MS Dhoni': '4a8a2e3b', 'N Rana': 'fb2d1dda', 'Nitin Menon': 'e1d41d9e', 'RA Jadeja': 'fe93fd9d', 

In [6]:
rows = []

for inning in data["innings"]:
    for inning_name, details in inning.items():
        team = details["team"]

        for delivery in details["deliveries"]:
            for ball, ball_data in delivery.items():

                over = int(ball)
                ball_num = int((ball - over) * 10)

                row = {
                    "over": over,
                    "ball": ball_num,
                    "batting_team": team,
                    "batter": ball_data.get("batsman"),
                    "bowler": ball_data.get("bowler"),
                    "runs": ball_data["runs"]["total"],
                    "extras": ball_data["runs"]["extras"],
                    "wicket": 1 if "wicket" in ball_data else 0
                }

                rows.append(row)

import pandas as pd
df = pd.DataFrame(rows)

df.head()

,over,ball,batting_team,batter,bowler,runs,extras,wicket
0,0,1,Chennai Super Kings,RD Gaikwad,UT Yadav,1,1,0
1,0,2,Chennai Super Kings,RD Gaikwad,UT Yadav,0,0,0
2,0,3,Chennai Super Kings,RD Gaikwad,UT Yadav,1,1,0
3,0,4,Chennai Super Kings,RD Gaikwad,UT Yadav,0,0,0
4,0,5,Chennai Super Kings,RD Gaikwad,UT Yadav,0,0,1


In [7]:
total_runs = df["runs"].sum() + df["extras"].sum()
print(total_runs)

274


In [8]:
batter_stats = df.groupby("batter").agg({
    "runs": "sum",
    "ball": "count",
    "wicket": "sum"
}).reset_index()

batter_stats.rename(columns={"ball": "balls_faced"}, inplace=True)

# Strike rate
batter_stats["strike_rate"] = (batter_stats["runs"] / batter_stats["balls_faced"]) * 100

print(batter_stats)

         batter  runs  balls_faced  wicket  strike_rate
0     AM Rahane    44           34       1   129.411765
1     AT Rayudu    15           17       0    88.235294
2     DP Conway     3            8       1    37.500000
3      MS Dhoni    51           38       0   134.210526
4        N Rana    25           17       1   147.058824
5     RA Jadeja    27           29       1    93.103448
6    RD Gaikwad     2            5       1    40.000000
7    RV Uthappa    30           23       1   130.434783
8        S Dube     3            6       1    50.000000
9    SP Jackson     3            3       0   100.000000
10      SS Iyer    20           19       0   105.263158
11  SW Billings    25           22       1   113.636364
12      VR Iyer    16           16       1   100.000000


In [9]:
bowler_stats = df.groupby("bowler").agg({
    "runs": "sum",
    "ball": "count",
    "wicket": "sum"
}).reset_index()

bowler_stats.rename(columns={"ball": "balls_bowled"}, inplace=True)

# Economy rate
bowler_stats["economy"] = bowler_stats["runs"] / (bowler_stats["balls_bowled"] / 6)

print(bowler_stats)

          bowler  runs  balls_bowled  wicket    economy
0     AD Russell    38            24       1   9.500000
1       AF Milne    19            15       0   7.600000
2       CV Varun    23            25       1   5.520000
3       DJ Bravo    20            24       3   5.000000
4     MJ Santner    31            24       1   7.750000
5      RA Jadeja    25            24       0   6.250000
6         S Dube    15             6       0  15.000000
7      SP Narine    15            24       1   3.750000
8    Shivam Mavi    35            26       0   8.076923
9   TU Deshpande    23            18       0   7.666667
10      UT Yadav    20            27       2   4.444444


In [10]:
team_stats = df.groupby("batting_team").agg({
    "runs": "sum",
    "wicket": "sum"
}).reset_index()

print(team_stats)

            batting_team  runs  wicket
0    Chennai Super Kings   131       5
1  Kolkata Knight Riders   133       4


In [11]:
df["total_runs"] = df["runs"] + df["extras"]

run_rate = df["total_runs"].sum() / (df["over"].max() + 1)

In [12]:
# Check structure
df.info()

# Check null values
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 237 entries, 0 to 236
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   over          237 non-null    int64 
 1   ball          237 non-null    int64 
 2   batting_team  237 non-null    object
 3   batter        237 non-null    object
 4   bowler        237 non-null    object
 5   runs          237 non-null    int64 
 6   extras        237 non-null    int64 
 7   wicket        237 non-null    int64 
 8   total_runs    237 non-null    int64 
dtypes: int64(6), object(3)
memory usage: 16.8+ KB
over            0
ball            0
batting_team    0
batter          0
bowler          0
runs            0
extras          0
wicket          0
total_runs      0
dtype: int64


In [13]:
# Fill missing numeric values with 0
df["runs"] = df["runs"].fillna(0)
df["extras"] = df["extras"].fillna(0)
df["wicket"] = df["wicket"].fillna(0)

# Drop rows where key info is missing
df = df.dropna(subset=["batter", "bowler", "batting_team"])

In [14]:
df["over"] = df["over"].astype(int)
df["ball"] = df["ball"].astype(int)
df["runs"] = df["runs"].astype(int)
df["extras"] = df["extras"].astype(int)
df["wicket"] = df["wicket"].astype(int)

In [15]:
df = df.drop_duplicates()

In [16]:
df["total_runs"] = df["runs"] + df["extras"]

In [21]:
df["ball_number"] = df["over"] * 6 + df["ball"]
df["ball_number"] = df["ball_number"].astype(int)

In [22]:
print(df.describe())
print(df["batting_team"].unique())

             over        ball        runs      extras      wicket  total_runs  \
count  201.000000  201.000000  201.000000  201.000000  201.000000  201.000000   
mean     9.253731    3.442786    1.144279    0.049751    0.039801    1.194030   
std      5.774106    1.645588    1.504686    0.327891    0.195979    1.592848   
min      0.000000    1.000000    0.000000    0.000000    0.000000    0.000000   
25%      4.000000    2.000000    0.000000    0.000000    0.000000    0.000000   
50%      9.000000    4.000000    1.000000    0.000000    0.000000    1.000000   
75%     14.000000    5.000000    1.000000    0.000000    0.000000    1.000000   
max     19.000000    6.000000    7.000000    4.000000    1.000000    8.000000   

       ball_number  
count   201.000000  
mean     58.965174  
std      34.630388  
min       1.000000  
25%      29.000000  
50%      59.000000  
75%      89.000000  
max     120.000000  
['Chennai Super Kings' 'Kolkata Knight Riders']


In [24]:
# Runs should not be negative
df = df[df["runs"] >= 0]

# Ball should be between 1–6
df = df[(df["ball"] >= 1) & (df["ball"] <= 6)]

# Over should be valid
df = df[df["over"] >= 0]

In [25]:
df.head()

,over,ball,batting_team,batter,bowler,runs,extras,wicket,total_runs,ball_number
0,0,1,Chennai Super Kings,RD Gaikwad,UT Yadav,1,1,0,2,1
1,0,2,Chennai Super Kings,RD Gaikwad,UT Yadav,0,0,0,0,2
2,0,3,Chennai Super Kings,RD Gaikwad,UT Yadav,1,1,0,2,3
3,0,4,Chennai Super Kings,RD Gaikwad,UT Yadav,0,0,0,0,4
4,0,5,Chennai Super Kings,RD Gaikwad,UT Yadav,0,0,1,0,5


In [26]:
threshold = 0.3

null_percent = df.isnull().mean()
cols_to_drop = null_percent[null_percent > threshold].index

df = df.drop(columns=cols_to_drop)

print("Dropped columns:", cols_to_drop)

Dropped columns: Index([], dtype='object')


In [27]:
import numpy as np

numeric_df = df.select_dtypes(include=np.number)
corr_matrix = numeric_df.corr()

print(corr_matrix)

                 over      ball      runs    extras    wicket  total_runs  \
over         1.000000 -0.032405  0.106260 -0.056878 -0.026643    0.088670   
ball        -0.032405  1.000000  0.058881  0.088701  0.084615    0.073882   
runs         0.106260  0.058881  1.000000  0.167796 -0.138260    0.979193   
extras      -0.056878  0.088701  0.167796  1.000000  0.046840    0.364361   
wicket      -0.026643  0.084615 -0.138260  0.046840  1.000000   -0.120965   
total_runs   0.088670  0.073882  0.979193  0.364361 -0.120965    1.000000   
ball_number  0.998872  0.015100  0.109102 -0.052687 -0.022633    0.092217   

             ball_number  
over            0.998872  
ball            0.015100  
runs            0.109102  
extras         -0.052687  
wicket         -0.022633  
total_runs      0.092217  
ball_number     1.000000  


In [28]:
threshold = 0.9

corr_matrix = numeric_df.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

to_drop = [column for column in upper.columns if any(upper[column] > threshold)]

df = df.drop(columns=to_drop)

print("Dropped due to correlation:", to_drop)

Dropped due to correlation: ['total_runs', 'ball_number']
